# Model Demo: FreshFruitsClassifier

This notebook demonstrates how to:

- Load a trained model (e.g., EfficientNet-B0)

- Run inference on a few test images from `data/processed`

- Visualize predictions

- Verify that the repository setup works end-to-end


In [ ]:
# Imports and configuration

import os

import torch

from torchvision import transforms

from PIL import Image



from src.models import get_model

from src.utils import load_checkpoint



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")


In [ ]:
# Load EfficientNet-B0 model and checkpoint

model_name = "efficientnet_b0"

num_classes = 2



model = get_model(model_name, num_classes=num_classes)

ckpt_path = os.path.join("..", "models", f"{model_name}_best.pth")



if os.path.exists(ckpt_path):

    model = load_checkpoint(model, ckpt_path, device)

    model.to(device)

    model.eval()

    print(f"Loaded checkpoint: {ckpt_path}")

else:

    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}. Please train the model first.")


In [ ]:
# Define transforms and helper for single image inference

image_size = 128



transform = transforms.Compose([

    transforms.Resize((image_size, image_size)),

    transforms.ToTensor(),

])



class_names = ["fresh", "spoiled"]



def predict_image(img_path: str):

    img = Image.open(img_path).convert("RGB")

    x = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():

        outputs = model(x)

        probs = torch.softmax(outputs, dim=1)[0]

        pred_idx = int(torch.argmax(probs).item())

    return img, class_names[pred_idx], probs.cpu().tolist()


In [ ]:
# Run prediction on a few test images and visualize

import glob

import matplotlib.pyplot as plt



test_dir_fresh = os.path.join("..", "data", "processed", "test", "fresh")

test_dir_spoiled = os.path.join("..", "data", "processed", "test", "spoiled")



image_paths = []

image_paths.extend(sorted(glob.glob(os.path.join(test_dir_fresh, "*.jpg")))[:4])

image_paths.extend(sorted(glob.glob(os.path.join(test_dir_spoiled, "*.jpg")))[:4])



fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes = axes.flatten()



for ax, img_path in zip(axes, image_paths):

    img, pred_label, probs = predict_image(img_path)

    ax.imshow(img)

    ax.set_title(f"Pred: {pred_label}")

    ax.axis("off")



plt.tight_layout()

plt.show()
